> **Purpose:** NJ ‖D−D̂‖₂, Q-criterion, RF distance vs p on balanced binary.  
> **Data:** generated

# Figure 8 — Classical NJ with subsampling

Three metrics versus the subsampling fraction `p`, then `p*` vs `n`:

1. $\|D - \hat D\|_2$  (distance-matrix spectral-norm error)
2. $Q$-criterion separation: distributions of $Q(i,j)$ for adjacent vs. non-adjacent leaf pairs at step 0 of NJ (Fig. 3 analog)
3. Robinson–Foulds distance from classical NJ on $\hat D$

Baseline: uniform sampling with **no** matrix completion. JC distance is $D = -\log R$ where $R$ is the JC similarity (LaTeX algorithm Step 2).

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json
from datetime import datetime
from pathlib import Path

_R = Path.cwd()
while _R.parent != _R and not (_R / "setup.py").exists():
    _R = _R.parent
_root = str(_R / "sub_sampled_fielder_vec")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Drop any stale cached copies of our project modules so this cell picks up
# edits made since the kernel started. autoreload handles subsequent edits.
for _mod in [m for m in list(sys.modules) if m.startswith(('scripts.', 'src.'))]:
    del sys.modules[_mod]

from src.config.presets import custom_config
from src.runners.nj_sweep import nj_sweep_for_params
from src.utils.nj_io import load_nj_results
from scripts.plot_nj_three_panel import _plot_three_panel
from scripts.plot_nj_p_star_vs_n import (
    _load_sweep, _compute_p_stars, _plot,
    _plot_overlay, _plot_q_overlay,
)

## Choose: load or launch
Set `RUN_DIR` to an existing sweep directory to **load** prior results. Leave it `None` to **launch** a fresh sweep with the grid below.

In [ ]:
# Default: load the existing mean-imputation sweep. Set RUN_DIR=None to
# launch a fresh sweep with the TAXA / SEQLEN / REPS / P_VALS grid below.
RUN_DIR = Path(_root) / 'results' / 'runs' / '20260514-174813-nj_sweep'

TAXA   = [128, 512, 1024, 2048]
SEQLEN = 2000
REPS   = 5
P_VALS = [1.0, 0.9, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]

if RUN_DIR is None:
    RUN_DIR = Path(_root) / 'results' / 'runs' / (datetime.now().strftime('%Y%m%d-%H%M%S') + '-nj_notebook')
    for n in TAXA:
        cfg = custom_config(num_taxa=n, sequence_length=SEQLEN, mutation_rate=0.1,
                            tree_model='balanced_binary', seq_model='JC69',
                            p_values=P_VALS, bootstrap_reps=REPS,
                            sampling_method='uniform', matrix_kind='distance')
        nj_sweep_for_params(cfg, n, SEQLEN, str(RUN_DIR / f'n{n}_L{SEQLEN}'))
else:
    RUN_DIR = Path(RUN_DIR)
print('sweep dir:', RUN_DIR)

## Three-panel plot, per n

In [ ]:
from IPython.display import Image, display
for sub in sorted(RUN_DIR.iterdir()):
    if not sub.is_dir():
        continue
    if not ((sub / 'nj_meta.json').exists() or (sub / 'nj_results.json').exists()):
        continue
    r = load_nj_results(sub)
    out = sub / 'nj_three_panel.png'
    _plot_three_panel(r, out)
    display(Image(filename=str(out)))

## p* vs n

In [ ]:
rows = _compute_p_stars(_load_sweep(RUN_DIR))

# Diagnostic-metric hint: the 2x2 overlay panel renders normalised metrics
# (L_inf and ||D - D_hat||_2 / ||D||_2) only when nj_meta_extra.json exists
# alongside each nj_meta.json. If none are found, _plot_overlay falls back to
# the legacy 1x2 layout — print the command to generate the extras.
_has_extras = any(p.exists() for p in RUN_DIR.glob('n*_L*/nj_meta_extra.json'))
if not _has_extras:
    print(f"[hint] no nj_meta_extra.json under {RUN_DIR}; the overlay will "
          f"render the legacy 1x2 layout. To add the normalised diagnostic "
          f"panels (L_inf and ||D - D_hat||_2 / ||D||_2), run:\n"
          f"    python scripts/nj_recompute_normalized_metrics.py {RUN_DIR}\n")

_plot(rows, RUN_DIR / 'nj_p_star_vs_n.png')
_plot_overlay(RUN_DIR, RUN_DIR / 'nj_overlay_specnorm_rf.png')
_plot_q_overlay(RUN_DIR, RUN_DIR / 'nj_overlay_q.png')

for name in ('nj_overlay_specnorm_rf.png', 'nj_overlay_q.png', 'nj_p_star_vs_n.png'):
    display(Image(filename=str(RUN_DIR / name)))
rows